<a href="https://colab.research.google.com/github/urmilapol/urmilapolprojects/blob/master/vasudevay.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install prerequisites & Devanagari fonts
!apt-get install -y fonts-deva fonts-deva-extra libraqm-dev -qq
!pip install Pillow pandas -q

import os
from PIL import Image, ImageDraw, ImageFont
from google.colab import files

# 2. Upload your Vishnu color template image (vishnu1.jpg)
print("Upload the Lord Vishnu color image:")
uploaded = files.upload()
img_name = list(uploaded.keys())[0]

# 3. Setup Devanagari Font
font_path = "/usr/share/fonts/truetype/fonts-deva-extra/gargi.ttf"
if not os.path.exists(font_path):
    font_path = "/usr/share/fonts/truetype/lohit-devanagari/Lohit-Devanagari.ttf"

# 5. Open Base Image & Calculate Scaling (moved outside loop)
base_img = Image.open(img_name).convert("RGBA")
W, H = base_img.size

# Scaled typography sizes based on resolution
font_title = ImageFont.truetype(font_path, int(W * 0.065))
font_shlok = ImageFont.truetype(font_path, int(W * 0.055))
font_meaning = ImageFont.truetype(font_path, int(W * 0.045))

# Define Colors
COLOR_TITLE = (45, 15, 5)        # Deep Antique Chestnut
COLOR_SHLOK = (105, 25, 10)      # Rich Crimson / Maroon
COLOR_TEXT = (25, 15, 10)        # Crisp Charcoal Ink

# Text Wrapping Function
def wrap_text_pixels(text, font, max_width, draw):
    lines = []
    paragraphs = text.split('\n')
    for p in paragraphs:
        words = p.strip().split()
        if not words:
            if lines and lines[-1] != '': # Add a blank line if a paragraph ends with a newline and it's not a consecutive blank line. This avoids extra blank lines if the input itself contains multiple newlines. # Lgtm [py/comparison-to-empty-string] # Lgtm [py/comparison-to-empty-string] # Lgtm [py/comparison-to-empty-string] # Lgtm [py/comparison-to-empty-string]
                lines.append('')
            continue
        cur_line = []
        for word in words:
            test_line = " ".join(cur_line + [word])
            bbox = draw.textbbox((0, 0), test_line, font=font, language='hi')
            if (bbox[2] - bbox[0]) <= max_width:
                cur_line.append(word)
            else:
                if cur_line:
                    lines.append(" ".join(cur_line))
                cur_line = [word]
        if cur_line:
            lines.append(" ".join(cur_line))
    return lines

# Bold Drawing Function
def draw_bold(draw, pos, text, font, fill, align_center=False, max_w=None):
    x, y = pos
    if align_center and max_w:
        bbox = draw.textbbox((0, 0), text, font=font, language='hi')
        text_w = bbox[2] - bbox[0]
        x = x + (max_w - text_w) // 2
    draw.text((x, y), text, font=font, fill=fill, language='hi')
    draw.text((x + 1, y), text, font=font, fill=fill, language='hi')
    draw.text((x, y + 1), text, font=font, fill=fill, language='hi')

language_val = 'hi'

# Loop through each shloka in df_csv to generate cards
for index, row in df_csv.iterrows():
    # 4. Text from Image (moved inside loop)
    shloka_num = str(row['Sr.No.'])
    sanskrit_shloka = row['Sanskrit Shlok']
    marathi_meaning = row['Marathi Meaning']

    # 6. Render Card
    card = base_img.copy()
    draw = ImageDraw.Draw(card)

    # Positioning on lower parchment
    margin_x = int(W * 0.11)
    content_w = W - (2 * margin_x)
    y = int(H * 0.7) # Adjusted to start lower on the image

    # 1. Title Heading
    title_text = f"श्लोक {shloka_num} चा भावार्थ:"
    draw_bold(draw, (margin_x, y), title_text, font=font_title, fill=COLOR_TITLE)
    y += int(W * 0.065 * 1.5) # Adjusted line spacing

    # 2. Sanskrit Shloka (Centered with deep maroon ink)
    shlok_lines = sanskrit_shloka.split('\n')
    for line in shlok_lines:
        draw_bold(draw, (margin_x, y), line.strip(), font=font_shlok, fill=COLOR_SHLOK, align_center=True, max_w=content_w)
        y += int(W * 0.055 * 1.4) # Adjusted line spacing

    y += int(W * 0.02) # Spacer

    # 3. Marathi Meaning
    meaning_lines = wrap_text_pixels(marathi_meaning, font_meaning, content_w, draw)
    for line in meaning_lines:
        draw_bold(draw, (margin_x, y), line, font=font_meaning, fill=COLOR_TEXT)
        y += int(W * 0.045 * 1.38) # Adjusted line spacing

    # Save Output
    output_path = f"vishnu_shlok_{shloka_num}_card.png"
    card.convert("RGB").save(output_path, "PNG", quality=95)
    print(f"Successfully generated {output_path}!")

print("All shloka images generated!")
# Removed files.download(output_path) from loop. User can download individually or in a zip later.

Upload the Lord Vishnu color image:
